# Optimize all three launch amplitudes at two fixed powers

Run the same three-mode fiber at **100 kW** and **81.943 kW**. In each run,
update all three real launch coefficients together and minimize the complex-field
distortion **over all three modes**. Peak power is fixed within each run.

$$D_{\rm NL}=1-\frac{|\sum_{m=1}^{3}\int A_{\rm lin,m}^*(L,t)A_{\rm NL,m}(L,t)\,dt|^2}
{(\sum_{m=1}^{3}\int|A_{\rm lin,m}|^2dt)(\sum_{m=1}^{3}\int|A_{\rm NL,m}|^2dt)}.$$

Use $0\le c_1,c_2,c_3\le100$, with $c_1+c_2+c_3=100$ to remove duplicate scalings.
**No individual coefficient is fixed.** The pulse constructor normalizes amplitudes,
so $P_m/P=c_m^2/\sum_jc_j^2$; the coefficients are not percentages of power.
Three adjustable coordinates describe two independent mixing ratios. All relative phases remain zero.

This follows the launch notebook's paired linear/nonlinear evaluator and the two reference
notebooks' projected gradient descent, backtracking, caches, validation, and mode plots.
It uses finite differences of DNL, not the squeezing notebook's photon-noise adjoint.
Run all cells in order with **GNLSE (Julia)** after changing settings.

In [ ]:
using Pkg
workspace = let dir = abspath(@__DIR__)
    while !isfile(joinpath(dir, "notebook_helpers", "run_library.py"))
        dirname(dir) == dir && error("Open this notebook inside the Nonlinear Optics workspace.")
        dir = dirname(dir)
    end
    dir
end
ROOT = get(ENV, "DNL_PROJECT_ROOT", joinpath(workspace, "FastDifferentiableMMNLO"))
Pkg.activate(ROOT)
using PulsePropagation, FFTW, LinearAlgebra, Printf, TOML, Dates, SHA, NPZ
using PyPlot

BLAS.set_num_threads(1)
FFTW.set_num_threads(1)
run_dir = haskey(ENV, "DNL_RUN_DIR") ? ENV["DNL_RUN_DIR"] :
    mktempdir(mkpath(joinpath(workspace, "output", "organized", "runs", "Modal amplitude coefficients", "Gradient descent")); prefix=Dates.format(now(), "yyyy-mm-dd") * "_step_index_dnl_three_coefficient_optimization_", cleanup=false)
mkpath(run_dir)
println("Results: ", run_dir)

# Load the launch, objective, descent, and plotting functions used below.
include(joinpath(workspace, "notebook_helpers", "julia", "Modal amplitude coefficients", "step_index_dnl_three_coefficient_optimization.jl"))

## 1. Controls, normalization, and budgets

Start both power cases at equal amplitudes. Internally use $u=c/100$ and project
each proposed vector onto $u_m\ge0$, $\sum_m u_m=1$. This allows every modal mixture
of nonnegative, equal-phase amplitudes without wasting steps on identical scalings.

The **0.5 coefficient perturbation** estimates derivatives; check it against 0.25.
The actual move comes from the learning rate and backtracking, with no coefficient
changing by more than 10 in an accepted step. Each three-component central gradient
usually needs six DNL evaluations, each with two forward propagations.

Each power case has **200 optimization evaluations** and **40 baseline/validation
evaluations**. Cached repeats are free; a failed new calculation still consumes budget.
There is no post-search simulation sweep: the triangle plots use saved evaluations.

In [ ]:
power_cases_W = [100_000.0, 81_943.0]
start_coefficients = fill(100.0/3, 3)
coefficient_bounds = (0.0, 100.0)
fd_step = 0.5                       # amplitude-coefficient units
learning_rate = 0.10                # update u = c/100
max_iterations = 20
max_optimizer_evaluations = 200     # per fixed-power case
max_check_evaluations = 40          # preflight + final checks, per case
max_coefficient_change = 10.0
gradient_tolerance = 1e-4           # infinity norm of projected gradient in u
gradient_check_atol = 1e-5
gradient_check_rtol = 0.01
minimum_brightness_fraction = 0.50
search_dz = 0.005                   # 5 mm
validation_dz = 0.001               # 1 mm
dnl_tolerance = 2e-4
minimum_decrease = 1e-10

settings = Dict(
    "control"=>"three nonnegative real amplitudes; sum(c)=100; zero relative phases",
    "objective"=>"all-three-mode matched linear/nonlinear complex-field DNL",
    "objective_modes"=>[1,2,3], "objective_filter"=>"none",
    "power_cases_W"=>power_cases_W, "start_coefficients"=>start_coefficients,
    "coefficient_bounds"=>collect(coefficient_bounds),
    "finite_difference_step_coefficient"=>fd_step,
    "learning_rate_u"=>learning_rate, "max_iterations"=>max_iterations,
    "max_optimizer_evaluations_per_power"=>max_optimizer_evaluations,
    "max_check_evaluations_per_power"=>max_check_evaluations,
    "max_coefficient_change"=>max_coefficient_change,
    "gradient_tolerance_u"=>gradient_tolerance,
    "gradient_check_atol"=>gradient_check_atol, "gradient_check_rtol"=>gradient_check_rtol,
    "minimum_brightness_fraction"=>minimum_brightness_fraction,
    "brightness_definition"=>"all-mode unfiltered photons relative to equal-three-mode fine baseline at the same power",
    "search_dz_m"=>search_dz, "validation_dz_m"=>validation_dz,
    "dnl_tolerance"=>dnl_tolerance,
    "armijo_c1"=>1e-4,"max_backtracking_halvings"=>12,
    "minimum_DNL_decrease"=>minimum_decrease,
    "Nt"=>2048, "validation_Nt"=>4096, "time_window_ps"=>10.0,
    "pulse_fwhm_ps"=>0.2, "fiber_length_m"=>2.86, "carrier_m"=>1379.2e-9,
    "core_radius_m"=>52.5e-6, "NA"=>0.1, "num_modes"=>3,
    "linear_reference"=>"same candidate input and fiber; n2 = 0",
    "cp_target_error"=>1e-3, "cp_seed"=>Int(0x4d4d474e),
    "julia_version"=>string(VERSION),
    "manifest_sha256"=>bytes2hex(sha256(read(joinpath(ROOT, "Manifest.toml")))),
)
for P in power_cases_W
    mkpath(case_dir(P))
end
write_toml(joinpath(run_dir, "settings.toml"), settings)
println("Fixed powers (kW): ", power_cases_W./1000)
println("Initial coefficients: ", start_coefficients)

## 2. Build the same fiber and CP representation once

Mode 1 is LP01; modes 2 and 3 are the cosine and sine LP11 orientations.
Changing launch amplitudes or peak power does not change the fiber's mode profiles,
dispersion, or overlap tensor. Reuse them for both power cases.

The linear and nonlinear branches differ only in `n2`. Both score and brightness now
include all three modes. The old modes-1+2 DNL is retained only as a diagnostic.

In [ ]:
properties = compute_fiber_properties(
    StepIndex(settings["carrier_m"], settings["core_radius_m"], settings["NA"]),
    Silica(), settings["num_modes"];
    polarization=:scalar, mode_basis=:LP, beta_order=5)
S_cp = cp_compress(properties.S; error=settings["cp_target_error"], seed=settings["cp_seed"])
context_cache = Dict{Tuple{Int,Symbol},Any}()
main_context = simulation_context(settings["Nt"])
time_grid, domain = main_context.grid, main_context.domain
mode_names = ("LP01", "LP11-cos", "LP11-sin")
@printf("CP rank: %d; relative tensor error: %.4g\n", cp_rank(S_cp), S_cp.relative_error)
println("Mode labels: ", properties.modes.labels)

## 3. Evaluate one launch and keep a complete ledger

`dnl_at(c, P)` normalizes the amplitude vector, propagates the same pulse through
both branches, and calculates the all-mode complex overlap. The complex modal
overlaps are summed **before** squaring, preserving relative phase information.

Cache keys include all three amplitudes, power, propagation step, time resolution,
and dense/CP representation. Each new attempted evaluation is counted before solving.
The all-zero vector is invalid; pure-mode launches are allowed.

In [ ]:
evaluation_cache = Dict{Any,NamedTuple}()
field_cache = Dict{Int,NamedTuple}()
evaluations, failures = NamedTuple[], NamedTuple[]
attempt_id = Ref(0)
used = Dict(P=>Dict(:optimization=>0, :validation=>0) for P in power_cases_W)

# Cheap checks of the constraints and the ALL-MODE overlap definition.
let
    for v in ([0.2,0.3,0.5],[-2.0,3.0,0.4],[1.0,1.0,1.0])
        u=project_simplex(v)
        @assert all(u .>= 0) && isapprox(sum(u),1.0;atol=1e-14)
        @assert isapprox(project_simplex(u),u;atol=1e-14)
    end
    @assert canonical_coefficients([20,30,50]) ≈ canonical_coefficients([40,60,100])
    a=zeros(ComplexF64,4,3,1); a[1,1,1]=1; a[1,2,1]=1
    b=copy(a); b[1,3,1]=2
    @assert field_fidelity(a,b)<1
    @assert isapprox(field_fidelity(a,2exp(0.7im).*a),1;atol=1e-12)
    for m in 1:3
        pure=zero(a); pure[1,m,1]=1
        @assert field_fidelity(pure,pure)==1
    end
end
println("Projection, scale-equivalence, and all-mode overlap checks passed.")

## 4. Estimate a three-component gradient

For component $i$, temporarily change only raw $c_i$ by $\pm0.5$ and evaluate
the whole normalized launch. The total peak power is unchanged. Each call recomputes
both propagation branches. Multiply $dD/dc_i$ by 100 to obtain $dD/du_i$.

The probe vectors need not sum to 100 before normalization. They define the smooth
scale-invariant objective around the current mixture. Accepted optimization states
are projected back onto the sum-100 triangle. At 0 or 100 use a second-order one-sided stencil.

In [ ]:
let
    for c in (0.0,0.2,50.0,99.8,100.0)
        points,weights=coefficient_stencil(c)
        @assert isapprox(sum(weights.*(points.-37).^2),2(c-37);atol=1e-9)
    end
    # Check the vector probes against an analytic scale-invariant objective.
    c=[20.0,30.0,50.0];u=c./100

    g=zeros(3)
    for (j,v,w) in gradient_probes(u;h=0.01)
        g[j]+=w*check_amplitude_objective(v)
    end
    analytic=100 .* (-2c[1]^2 .* c ./ sum(abs2,c)^2)
    analytic[1]+=100*2c[1]/sum(abs2,c)
    @assert isapprox(g,analytic;rtol=1e-6,atol=1e-8)
end
println("Interior, boundary, and vector-gradient formula checks passed.")

## 5. Check both starting points before searching

Compare 5 mm and 1 mm DNL and the gradients from coefficient steps 0.5 and 0.25.
Use an absolute-plus-relative gradient tolerance so a nearly flat direction is not
judged by a misleading relative error. Define each power's brightness floor from
its own fine, equal-three-mode baseline.

In [ ]:
preflights=Dict{Float64,Any}()
brightness_floors=Dict{Float64,Float64}()
for P in power_cases_W
    baseline=dnl_at(start_coefficients,P)
    fine=dnl_at(start_coefficients,P;dz=validation_dz)
    u=canonical_coefficients(start_coefficients)./100
    g=launch_gradient(u,P)
    gh=launch_gradient(u,P;h=fd_step/2)
    agreement=gradient_agreement(g,gh)
    step_error=abs(baseline.D_NL-fine.D_NL)
    ready=agreement.passed && step_error<=dnl_tolerance
    brightness_floors[P]=minimum_brightness_fraction*fine.photons
    preflights[P]=(;baseline,fine,g,gh,agreement,step_error,ready)
    write_toml(joinpath(case_dir(P),"preflight.toml"),Dict(
        "power_W"=>P,"ready"=>ready,"baseline_DNL_search"=>baseline.D_NL,
        "baseline_DNL_fine"=>fine.D_NL,"step_error"=>step_error,
        "gradient_u"=>g,"half_step_gradient_u"=>gh,
        "gradient_difference"=>agreement.error,"gradient_check_threshold"=>agreement.threshold,
        "brightness_floor"=>brightness_floors[P]))
    @printf("P=%.3f kW: step error %.3g; gradient difference %.3g / tolerance %.3g; ready=%s\n",
        P/1000,step_error,agreement.error,agreement.threshold,string(ready))
end
@assert all(r.ready for r in values(preflights)) "Refine search_dz or fd_step before optimizing."

## 6. Update the full vector with projected gradient descent

At the current launch, estimate all three derivatives. Propose
$v=\Pi(u-0.10\nabla D)$, where $\Pi$ projects onto the nonnegative sum-one triangle.
Cap the largest coefficient move at 10. Backtracking repeatedly halves this move
until DNL decreases sufficiently and the all-mode brightness floor is met.

Stop at 20 accepted iterations, the 200-evaluation budget, a small projected gradient,
or an unsuccessful line search. A stop reason is not by itself evidence of a minimum;
the next section checks the selected point on the finer grid.

In [ ]:
optimizations=Dict(P=>optimize_launch(P) for P in power_cases_W)

## 7. Validate each selected launch and compare like with like

For each power, compare 5, 2.5, and 1 mm propagation steps; check the fine-grid
gradient at perturbations 0.5 and 0.25; probe four nearby feasible directions;
compare dense and CP overlap tensors; and double the time samples to 4096.
Recompute the original `[50,50,0]` launch and the earlier two-mode selected launch
using the **new all-three-mode objective**. Their earlier modes-1+2 scores are not comparable.

In a circular scalar fiber the two LP11 modes are rotated versions of one another.
At the equal-amplitude baseline, compare three orientations with the same LP01
power and total LP11 power. A small orientation-dependent score can be a grid/CP
effect. All three coordinates remain free; this check does not freeze an orientation.
Because these are rotated versions of the same LP11 shape, changing their ratio can
mainly rotate the beam. Nearly overlapping coefficient traces do not mean a coefficient
was held fixed: symmetry makes the two orientations respond almost equally.

Report numerical agreement, improvement, and fine-grid stationarity separately.
These checks do not establish a global minimum or convergence with additional fiber modes.

In [ ]:
# Rounded coefficients reported by the existing two-mode launch notebooks.
previous_two_mode=Dict(100_000.0=>[37.577380,62.422620,0.0],
    81_943.0=>[1.545451,98.454549,0.0])
validations=Dict{Float64,Any}()
summaries=Dict{Float64,Any}()
comparison_rows=NamedTuple[]
for P in power_cases_W
    opt=optimizations[P]
    base_fine=preflights[P].fine
    c=coefficients(opt.best);u=c./100
    best_fine=dnl_at(c,P;dz=validation_dz)
    best_mid=dnl_at(c,P;dz=search_dz/2)
    old_baseline=dnl_at([50.0,50.0,0.0],P;dz=validation_dz)
    old_selected=dnl_at(previous_two_mode[P],P;dz=validation_dz)
    g=launch_gradient(u,P;dz=validation_dz)
    gh=launch_gradient(u,P;h=fd_step/2,dz=validation_dz)
    agreement=gradient_agreement(g,gh)
    pg=norm(projected_gradient(u,gh),Inf)

    # Two independent directions in the sum-one plane, each with both signs.
    neighbor_rows=NamedTuple[]
    for direction in ([1.0,-1.0,0.0],[0.5,0.5,-1.0]), sign in (-1.0,1.0)
        nearby=100 .* project_simplex(u.+sign.*(fd_step/100).*direction)
        push!(neighbor_rows,dnl_at(nearby,P;dz=search_dz/2))
    end
    save_rows(joinpath(case_dir(P),"neighbors.csv"),neighbor_rows)
    feasible_neighbors=filter(r->r.photons>=brightness_floors[P],neighbor_rows)
    neighbor_lower=any(r->r.D_NL<best_mid.D_NL-minimum_decrease,feasible_neighbors)
    neighbor_gain=isempty(feasible_neighbors) ? 0.0 : max(0.0,best_mid.D_NL-minimum(r.D_NL for r in feasible_neighbors))

    dense_base=dnl_at(start_coefficients,P;dz=validation_dz,representation=:dense)
    dense_best=dnl_at(c,P;dz=validation_dz,representation=:dense)
    time_base=dnl_at(start_coefficients,P;dz=validation_dz,nt=settings["validation_Nt"])
    time_best=dnl_at(c,P;dz=validation_dz,nt=settings["validation_Nt"])
    a=canonical_coefficients(start_coefficients)
    lp11=hypot(a[2],a[3])
    rotation1=dnl_at(canonical_coefficients([a[1],lp11,0.0]),P;dz=validation_dz)
    rotation2=dnl_at(canonical_coefficients([a[1],0.0,lp11]),P;dz=validation_dz)

    dz_error=max(abs(opt.best.D_NL-best_fine.D_NL),abs(best_mid.D_NL-best_fine.D_NL))
    cp_error=max(abs(dense_base.D_NL-base_fine.D_NL),abs(dense_best.D_NL-best_fine.D_NL))
    time_error=max(abs(time_base.D_NL-base_fine.D_NL),abs(time_best.D_NL-best_fine.D_NL))
    rotation_scores=[base_fine.D_NL,rotation1.D_NL,rotation2.D_NL]
    rotation_error=maximum(rotation_scores)-minimum(rotation_scores)
    history_ok=all(r->all(x->0<=x<=100,coefficients(r)) &&
        isapprox(sum(coefficients(r)),100;atol=1e-10),opt.history) &&
        all(diff([r.D_NL for r in opt.history]).<0) &&
        all(r->r.max_coefficient_step<=max_coefficient_change+1e-10,opt.history)
    checks=Dict(
        "preflight"=>preflights[P].ready,
        "accepted_path_constraints"=>history_ok,
        "step_size_agreement"=>dz_error<=dnl_tolerance,
        "fine_gradient_step_agreement"=>agreement.passed,
        "cp_dense_agreement"=>cp_error<=dnl_tolerance,
        "time_resolution_agreement"=>time_error<=dnl_tolerance,
        "LP11_rotation_agreement"=>rotation_error<=dnl_tolerance,
        "fine_brightness"=>best_fine.photons>=brightness_floors[P])
    numerical_ok=all(values(checks))
    stationary=agreement.passed && pg<=gradient_tolerance
    improvement=base_fine.D_NL-best_fine.D_NL
    verified_improvement=numerical_ok && improvement>dnl_tolerance &&
        dense_best.D_NL<dense_base.D_NL && time_best.D_NL<time_base.D_NL
    summary=Dict(
        "power_W"=>P,"objective_modes"=>[1,2,3],"stop_reason"=>opt.status,
        "accepted_steps"=>length(opt.history)-1,"selected_coefficients"=>c,
        "selected_input_power_fractions"=>c.^2 ./ sum(abs2,c),
        "baseline_DNL_fine"=>base_fine.D_NL,"selected_DNL_fine"=>best_fine.D_NL,
        "absolute_DNL_improvement"=>improvement,
        "fractional_DNL_reduction"=>improvement/base_fine.D_NL,
        "brightness_ratio"=>best_fine.photons/base_fine.photons,
        "max_step_size_DNL_difference"=>dz_error,
        "max_cp_dense_DNL_difference"=>cp_error,
        "max_time_resolution_DNL_difference"=>time_error,
        "LP11_baseline_rotation_DNL_range"=>rotation_error,
        "fine_gradient_u"=>g,"fine_half_step_gradient_u"=>gh,
        "fine_gradient_difference"=>agreement.error,
        "fine_gradient_check_threshold"=>agreement.threshold,
        "fine_projected_gradient_norm"=>pg,"projected_gradient_tolerance"=>gradient_tolerance,
        "lower_feasible_neighbor_found"=>neighbor_lower,"largest_neighbor_DNL_gain"=>neighbor_gain,
        "numerical_checks_passed"=>numerical_ok,"verified_improvement"=>verified_improvement,
        "fine_grid_stationary"=>stationary,
        "global_minimum_established"=>false,"additional_mode_convergence_checked"=>false,
        "new_optimization_evaluations"=>used[P][:optimization],
        "new_check_evaluations"=>used[P][:validation],
        "failed_evaluations"=>count(r->r.power_W==P,failures),
        "checks"=>Dict(k=>(v ? "pass" : "fail") for (k,v) in checks))
    summaries[P]=summary
    validations[P]=(;base_fine,best_fine,best_mid,old_baseline,old_selected,dense_base,dense_best,
        time_base,time_best,rotation1,rotation2,g,gh,agreement,pg,neighbor_rows)
    for (label,row) in (("equal_three_mode_baseline",base_fine),("three_coefficient_selected",best_fine),
                       ("original_two_mode_baseline",old_baseline),("previous_two_mode_selected",old_selected))
        push!(comparison_rows,comparison_row(label,row,base_fine))
    end
    write_toml(joinpath(case_dir(P),"summary.toml"),summary)
    save_rows(joinpath(case_dir(P),"comparisons.csv"),filter(r->r.power_W==P,comparison_rows))
    @printf("%.3f kW: fine DNL %.8f -> %.8f; checks=%s, stationary=%s, stop=%s\n",
        P/1000,base_fine.D_NL,best_fine.D_NL,string(numerical_ok),string(stationary),opt.status)
    @printf("  selected %s; optimization %d/%d, checks %d/%d\n",coeff_label(c),
        used[P][:optimization],max_optimizer_evaluations,used[P][:validation],max_check_evaluations)
end
save_rows(joinpath(run_dir,"comparisons.csv"),comparison_rows)
write_toml(joinpath(run_dir,"summary.toml"),Dict(power_label(P)=>summaries[P] for P in power_cases_W))

## 8. Watch the search move through the three-mode mixtures

Each row is one fixed power. Read left to right: the score, all three amplitude
coefficients, and the corresponding **input power fractions**. The triangle below
places pure LP01, LP11-cos, and LP11-sin at its corners. Dots are evaluations that
were actually computed; the line joins accepted steps. This is not a surface fit or sweep.

In [ ]:
mode_colors=("#366b9e","#cf7c32","#439576")
mode_styles=("-","-","--")
mode_markers=("o","s","none")
fig,axes=subplots(2,3;figsize=(16,8.4),constrained_layout=true)
for (row_index,P) in enumerate(power_cases_W)
    opt=optimizations[P];h=opt.history
    iterations=[r.iteration for r in h]
    axes[row_index,1].plot(iterations,[r.D_NL for r in h],"o-";
        color=mode_colors[1],markersize=4,label="Accepted path: 5 mm")
    axes[row_index,1].scatter([0,iterations[end]],
        [validations[P].base_fine.D_NL,validations[P].best_fine.D_NL];
        color="black",marker="x",s=60,label="Fine check: 1 mm",zorder=5)
    axes[row_index,1].set_ylabel("All-three-mode DNL")
    axes[row_index,1].set_title(@sprintf("%.3f kW | %s",P/1000,replace(opt.status,"_"=>" "));fontsize=10)
    axes[row_index,1].legend(;fontsize=8)
    for m in 1:3
        axes[row_index,2].plot(iterations,[coefficients(r)[m] for r in h];
            color=mode_colors[m],linestyle=mode_styles[m],marker=mode_markers[m],markersize=3,label=mode_names[m])
        axes[row_index,3].plot(iterations,[100coefficients(r)[m]^2/sum(abs2,coefficients(r)) for r in h];
            color=mode_colors[m],linestyle=mode_styles[m],marker=mode_markers[m],markersize=3,label=mode_names[m])
    end
    axes[row_index,2].set_ylabel("Amplitude coefficient (sum = 100)")
    axes[row_index,3].set_ylabel("Input modal power (%)")
    for j in 2:3
        axes[row_index,j].set_ylim(-2,102)
        axes[row_index,j].legend(;fontsize=8)
    end
    for j in 1:3
        axes[row_index,j].set_xlabel("Accepted iteration")
        tidy_axis(axes[row_index,j])
    end
end
fig.suptitle("Three adjustable launch amplitudes; bounded search results")
finish_figure(fig,joinpath(run_dir,"optimization_history"))
fig,axes=subplots(1,2;figsize=(13,5.7),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    ax=axes[j]
    rows=filter(r->r.power_W==P && r.dz==search_dz && r.nt==settings["Nt"] && r.representation=="cp",evaluations)
    xy=[triangle_xy(coefficients(r)) for r in rows]
    dots=ax.scatter(first.(xy),last.(xy);c=[r.D_NL for r in rows],cmap="viridis",s=18,alpha=0.8)
    path=[triangle_xy(coefficients(r)) for r in optimizations[P].history]
    ax.plot(first.(path),last.(path),"o-";color="#ca584b",markersize=3,label="Accepted steps",zorder=5)
    ax.scatter([first(path)[1]],[first(path)[2]];marker="o",s=85,
        facecolors="white",edgecolors="black",label="Equal-amplitude start",zorder=6)
    ax.scatter([last(path)[1]],[last(path)[2]];marker="*",s=155,
        color="#e79b36",edgecolors="black",label="Selected",zorder=7)
    ax.plot([0,1,0.5,0],[0,0,sqrt(3)/2,0];color="#888888",linewidth=1)
    ax.text(-0.03,-0.035,"LP01";ha="center",fontsize=10)
    ax.text(1.03,-0.035,"LP11-cos";ha="center",fontsize=10)
    ax.text(0.5,sqrt(3)/2+0.025,"LP11-sin";ha="center",fontsize=10)
    ax.set_aspect("equal");ax.set_xlim(-0.13,1.13);ax.set_ylim(-0.12,1.0)
    ax.axis("off")
    ax.set_title(@sprintf("%.3f kW: mixture by input power fraction",P/1000))
    ax.legend(;loc="upper left",fontsize=7.5)
    fig.colorbar(dots;ax,shrink=0.65,label="Sampled all-mode DNL (5 mm)")
end
finish_figure(fig,joinpath(run_dir,"sampled_mixture_triangles"))

## 9. Compare the launch choices and the output pulses

Every comparison below uses the 1 mm grid and the same all-mode DNL definition.
The previous two-mode launches are reference points, not new optimization results.
For each power, display the input, matched linear reference, and nonlinear output
at the equal-three-mode baseline and at the selected launch.

In [ ]:
fig,axes=subplots(1,2;figsize=(13,4.8),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    v=validations[P]
    rows=(v.base_fine,v.best_fine,v.old_baseline,v.old_selected)
    labels=["Equal three\nmode start","Three coefficients\nselected","Original\n[50, 50, 0]","Earlier two\nmode selected"]
    bars=axes[j].bar(1:4,[r.D_NL for r in rows];color=["#a8b6c6","#366b9e","#e5bf98","#cf7c32"])
    for k in 1:4
        axes[j].text(k,rows[k].D_NL+0.012,@sprintf("%.4f",rows[k].D_NL);ha="center",fontsize=9)
    end
    axes[j].set_xticks(1:4,labels;fontsize=8)
    axes[j].set_ylim(0,min(1.05,1.17maximum(r.D_NL for r in rows)+0.02))
    axes[j].set_ylabel("All-three-mode DNL (1 mm)")
    axes[j].set_title(@sprintf("%.3f kW",P/1000));tidy_axis(axes[j])
end
finish_figure(fig,joinpath(run_dir,"launch_comparisons"))

times_ps=collect(time_axis(time_grid))
wavelengths_nm=collect(wavelength_axis(time_grid,properties.omega0)).*1e9
wavelength_indices=findall(isfinite.(wavelengths_nm).&(wavelengths_nm.>0))
wavelength_indices=wavelength_indices[sortperm(wavelengths_nm[wavelength_indices])]
curve_labels=("Input","Linear reference","Nonlinear output")
curve_colors=("#777777","#cf7c32","#366b9e")
curve_styles=(":","--","-")
for P in power_cases_W
    plot_modal_output("baseline",validations[P].base_fine)
    plot_modal_output("selected",validations[P].best_fine)
end

## 10. See the spatial modes and where the energy goes

The spatial mode shapes are fixed by the fiber. Their signed field amplitudes are
shown below. The bars show how the total pulse energy is distributed among modes;
these fractions can change during nonlinear propagation.

In [ ]:
x,y=properties.modes.x,properties.modes.y
modes=properties.modes.fields # y, x, mode
let fig=figure(;figsize=(15,4.8),constrained_layout=true)
    for m in 1:3
        ax=subplot(1,3,m)
        profile=real.(modes[:,:,m]);profile ./= maximum(abs,profile)
        mesh=ax.pcolormesh(x.*1e6,y.*1e6,profile;cmap="seismic",vmin=-1,vmax=1,
            shading="auto",rasterized=true)
        theta=range(0,2pi;length=181)
        ax.plot(settings["core_radius_m"].*1e6.*cos.(theta),settings["core_radius_m"].*1e6.*sin.(theta);
            color="#555555",linestyle="--",linewidth=0.8)
        ax.set_title("Mode $m: "*mode_names[m]);ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)")
        ax.set_aspect("equal")
        fig.colorbar(mesh;ax,shrink=0.75,label="Relative field amplitude")
    end
    fig.suptitle("Fixed spatial modes; dashed circle marks the fiber core")
    finish_figure(fig,joinpath(run_dir,"spatial_modes"))
end
npzwrite(joinpath(run_dir,"spatial_mode_fields.npz"),Dict("x_m"=>x,"y_m"=>y,"modes_y_x_m"=>modes))
modal_rows=NamedTuple[]
for P in power_cases_W
    fig,axes=subplots(1,2;figsize=(12.5,4.5),constrained_layout=true,sharey=true)
    for (case_index,(label,row)) in enumerate((("Baseline",validations[P].base_fine),("Selected",validations[P].best_fine)))
        saved=field_cache[row.id]
        for (j,a) in enumerate((saved.input,saved.linear,saved.nonlinear))
            energy=vec(sum(abs2.(a);dims=(1,3)));fractions=energy./sum(energy)
            axes[case_index].bar(collect(1:3).+(j-2)*0.23,100fractions;
                width=0.23,color=curve_colors[j],label=curve_labels[j])
            for m in 1:3
                push!(modal_rows,(;power_W=P,case=lowercase(label),branch=curve_labels[j],mode=m,energy_fraction=fractions[m]))
            end
        end
        axes[case_index].set_xticks(1:3,collect(mode_names))
        axes[case_index].set_ylim(0,105);axes[case_index].set_ylabel("Fraction of total pulse energy (%)")
        axes[case_index].set_title(label*" "*coeff_label(coefficients(row));fontsize=10)
        axes[case_index].legend(;fontsize=8);tidy_axis(axes[case_index])
    end
    fig.suptitle(@sprintf("%.3f kW: modal energy fractions",P/1000))
    finish_figure(fig,joinpath(case_dir(P),"modal_energy_fractions"))
end
save_rows(joinpath(run_dir,"modal_energy_fractions.csv"),modal_rows)

## 11. Reconstruct the whole beam, including interference

At the total pulse peak, add the **complex** modal fields and then square the result.
This retains interference between modes. Each branch uses its own peak time.
All beam maps below share one absolute intensity scale across both powers.
The spatial integral is checked against the summed modal peak power.

Individual-mode maps in the next cell use each mode's own peak time. They show
each mode separately and cannot be added to recover the coherent whole-beam map.

In [ ]:
beam_maps=Dict{Tuple{Float64,String},Any}()
individual_maps=NamedTuple[]
for P in power_cases_W, (case_index,(label,row)) in enumerate((("baseline",validations[P].base_fine),("selected",validations[P].best_fine)))
    saved=field_cache[row.id]
    beam_maps[(P,label)]=[peak_beam_map(a) for a in (saved.input,saved.linear,saved.nonlinear)]
    fractions=[row.mode1_energy_fraction,row.mode2_energy_fraction,row.mode3_energy_fraction]
    for m in 1:3
        power_t=vec(sum(abs2.(saved.nonlinear[:,m,:]);dims=2));k=argmax(power_t)
        intensity=abs2.(modes[:,:,m]).*power_t[k]./1e12
        push!(individual_maps,(;power_W=P,case_index,label,mode=m,intensity,
            peak_power_W=power_t[k],time_ps=times_ps[k],energy_fraction=fractions[m]))
    end
end
observed_max=max(maximum(maximum(m.intensity) for maps in values(beam_maps) for m in maps),
    maximum(maximum(m.intensity) for m in individual_maps))
beam_color_max=max(100.0,20ceil(observed_max/20))
beam_color_ticks=collect(range(0,beam_color_max;length=6))
write_toml(joinpath(run_dir,"beam_plot_settings.toml"),Dict(
    "shared_min_W_per_um2"=>0.0,"shared_max_W_per_um2"=>beam_color_max,
    "observed_max_W_per_um2"=>observed_max))
for P in power_cases_W
    fig,axes=subplots(2,3;figsize=(15,8.6),constrained_layout=true)
    meshes=[]
    for (i,(label,row)) in enumerate((("baseline",validations[P].base_fine),("selected",validations[P].best_fine)))
        maps=beam_maps[(P,label)]
        for j in 1:3
            ax=axes[i,j];m=maps[j]
            mesh=ax.pcolormesh(x.*1e6,y.*1e6,m.intensity;cmap="magma",vmin=0,vmax=beam_color_max,
                shading="auto",rasterized=true)
            push!(meshes,mesh)
            ax.set_title(@sprintf("%s: %s\npeak %.2f kW at %.3f ps",label,curve_labels[j],m.peak_power_W/1000,m.peak_time_ps);fontsize=10)
            ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)");ax.set_aspect("equal")
        end
        npzwrite(joinpath(case_dir(P),label*"_beam_maps.npz"),Dict(
            "x_um"=>x.*1e6,"y_um"=>y.*1e6,
            "input_W_per_um2"=>maps[1].intensity,"linear_W_per_um2"=>maps[2].intensity,
            "nonlinear_W_per_um2"=>maps[3].intensity,
            "peak_times_ps"=>[m.peak_time_ps for m in maps],"peak_powers_W"=>[m.peak_power_W for m in maps],
            "shared_color_max_W_per_um2"=>beam_color_max,"launch_coefficients"=>coefficients(row)))
    end
    fig.colorbar(meshes[end];ax=vec(axes),shrink=0.8,ticks=beam_color_ticks,
        label="Intensity at each pulse's peak (W/μm²)")
    fig.suptitle(@sprintf("%.3f kW: coherent whole beam | start %s | selected %s",
        P/1000,coeff_label(start_coefficients),coeff_label(coefficients(validations[P].best_fine)));fontsize=11)
    finish_figure(fig,joinpath(case_dir(P),"baseline_selected_beams"))
end
println("Spatial integrals match summed modal peak powers for all displayed whole-beam maps.")

In [ ]:
for P in power_cases_W
    fig,axes=subplots(2,3;figsize=(15,8.6),constrained_layout=true)
    meshes=[]
    for m in filter(r->r.power_W==P,individual_maps)
        ax=axes[m.case_index,m.mode]
        mesh=ax.pcolormesh(x.*1e6,y.*1e6,m.intensity;cmap="magma",vmin=0,vmax=beam_color_max,
            shading="auto",rasterized=true)
        push!(meshes,mesh)
        ax.set_title(@sprintf("%s: %s\nenergy %.3g%% | peak %.3g kW at %.3f ps",
            m.label,mode_names[m.mode],100m.energy_fraction,m.peak_power_W/1000,m.time_ps);fontsize=10)
        ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)");ax.set_aspect("equal")
    end
    fig.colorbar(meshes[end];ax=vec(axes),shrink=0.8,ticks=beam_color_ticks,
        label="Single-mode peak intensity (W/μm²)")
    fig.suptitle(@sprintf("%.3f kW: individual nonlinear output modes",P/1000))
    finish_figure(fig,joinpath(case_dir(P),"baseline_selected_output_modes"))
end

## 12. Read the result without confusing improvement with convergence

`verified_improvement` means the selected launch beats its equal-three-mode baseline
and passes the numerical checks above. `fine_grid_stationary` additionally asks whether
the projected fine-grid gradient is below the chosen tolerance. A lower nearby probe
is reported separately. An iteration or evaluation limit leaves an unfinished search;
it is not a proof of optimality.

If doubling `Nt` changes DNL by more than `dnl_tolerance`, time resolution is not
settled at that tolerance. A CP/dense failure calls for a tighter compression error
or a dense rerun. Disagreement between the 0.5 and 0.25 gradients calls for smaller
perturbations before claiming stationarity. The notebook records these failures;
it does not automatically expand the evaluation budgets or relax the tolerances.

CSV ledgers include every new successful evaluation, rejected line-search candidates,
accepted states, and comparisons. TOML files store settings, checks, and stop reasons;
NPZ files store the displayed fields. The objective uses three modeled modes and zero
launch phase differences; neither more modes nor phase controls are optimized here.

In [ ]:
for P in power_cases_W
    s=summaries[P]
    println("\n",power_label(P))
    println("  coefficients: ",coeff_label(s["selected_coefficients"]))
    fractions=s["selected_input_power_fractions"]
    @printf("  input power fractions: [%.3f%%, %.3f%%, %.3f%%]\n",100fractions[1],100fractions[2],100fractions[3])
    @printf("  all-mode fine DNL: %.8f -> %.8f (%.2f%% reduction)\n",
        s["baseline_DNL_fine"],s["selected_DNL_fine"],100s["fractional_DNL_reduction"])
    println("  stop: ",s["stop_reason"],"; accepted steps: ",s["accepted_steps"])
    println("  numerical checks passed: ",s["numerical_checks_passed"],
        "; verified improvement: ",s["verified_improvement"],"; fine-grid stationary: ",s["fine_grid_stationary"])
    failed_checks=sort([name for (name,status) in s["checks"] if status=="fail"])
    if !isempty(failed_checks)
        println("  PRELIMINARY: failed checks = ",join(failed_checks,", "))
    end
    @printf("  max DNL differences: dz %.3g; CP/dense %.3g; Nt %.3g (tolerance %.3g)\n",
        s["max_step_size_DNL_difference"],s["max_cp_dense_DNL_difference"],
        s["max_time_resolution_DNL_difference"],dnl_tolerance)
    @printf("  projected gradient: %.6g (tolerance %.6g); lower neighbor: %s\n",
        s["fine_projected_gradient_norm"],gradient_tolerance,string(s["lower_feasible_neighbor_found"]))
    @printf("  new evaluations: optimization %d/%d; preflight + checks %d/%d\n",
        used[P][:optimization],max_optimizer_evaluations,used[P][:validation],max_check_evaluations)
end
println("\nResults: ",run_dir)